In [249]:
import xarray as xr 
from anemoi.datasets import open_dataset
import numpy as np
import yaml
import os 
from pathlib import Path

### Variables

In [274]:
ds = xr.open_zarr("cerra_jan2024_boz.zarr")

print(ds.dims)
print(ds.coords)

for name in ("level", "levelist", "isobaricInhPa", "plev", "pressure"):
    if name in ds.coords:
        print("Pressure coord:", name, ds.coords[name].values)
for var in ("u","v","t","z","r"):
    if var in ds:
        print(var, ds[var].dims)

FrozenMappingWarningOnValuesAccess({'time': 248, 'y': 157, 'x': 211, 'level': 9})
Coordinates:
    entireAtmosphere  float64 8B ...
    latitude          (y, x) float64 265kB dask.array<chunksize=(157, 211), meta=np.ndarray>
  * level             (level) float64 72B 1e+03 950.0 900.0 ... 600.0 500.0
    longitude         (y, x) float64 265kB dask.array<chunksize=(157, 211), meta=np.ndarray>
    meanSea           float64 8B ...
    step              timedelta64[ns] 8B ...
    surface           float64 8B ...
  * time              (time) datetime64[ns] 2kB 2024-01-01 ... 2024-01-31T21:...
    valid_time        datetime64[ns] 8B ...
Pressure coord: level [1000.  950.  900.  850.  800.  750.  700.  600.  500.]
u ('time', 'level', 'y', 'x')
v ('time', 'level', 'y', 'x')
t ('time', 'level', 'y', 'x')
z ('time', 'level', 'y', 'x')
r ('time', 'level', 'y', 'x')


In [275]:
atmospheric_dims = ("time", "level","y", "x")
atmospheric_fields = [var for var in ds.data_vars if ds[var].dims == atmospheric_dims]
for var in atmospheric_fields:
    print(f" - {var}")

 - r
 - t
 - u
 - v
 - z


In [276]:
surface_dims = ("time", "y", "x")
surface_fields = [var for var in ds.data_vars if ds[var].dims == surface_dims]
for var in surface_fields:
    print(f" - {var}")

 - mcc
 - msl
 - synthetic_windpower
 - t2m
 - wdir10_cos
 - wdir10_sin
 - wdir_cos100
 - wdir_cos150
 - wdir_cos200
 - wdir_cos50
 - wdir_sin100
 - wdir_sin150
 - wdir_sin200
 - wdir_sin50
 - ws10
 - ws100
 - ws150
 - ws200
 - ws50


In [277]:
constant_dims = ("y","x")
constant_fields = [var for var in ds.data_vars if ds[var].dims == constant_dims]
for var in constant_fields:
    print(f" - {var}")

 - lsm
 - surface_geopotential


In [278]:
recipe_keys = [
    "name",
    "description",
    "dates",
    "input",
    "build"
]
recipe = dict.fromkeys(recipe_keys)

In [279]:
recipe["name"] = "Cerra-BOZ-Jan2024-3h-v0"
recipe["description"] = "Toy dataset for wind power prediction in BOZ"

In [280]:
url = "cerra_jan2024_boz.zarr"

In [281]:
# Set the dates part of the config
dates = {
    "start": "2024-01-01T00:00:00",
    "end": "2024-01-31T21:00:00",
    "frequency": "3h"
}

# and add it to the config
recipe["dates"] = dates

In [282]:
surface_variables = {
    "xarray-zarr": {                    # The key defining the type, the value is another dictionary with options
        "url": url,                     # The URL of the dataset
        "param": surface_fields,    # The variables to extract                   
    }
}

In [283]:
pressure_levels=[500,600,700,750,800,850,900,950,1000]
upper_air_variables = {
    "xarray-zarr": {                    # The key defining the type, the value is another dictionary with options
        "url": url,                     # The URL of the dataset
        "param": atmospheric_fields,    # The variables to extract                   
    }
}

In [284]:
constant_variables = { 
    "repeated-dates": {      # The key defining the type.
        "mode": "constant",  # The mode, defining here that the fields are constant, other options (climatology, closest) exist
        "source": {          # Which source to use for the constant fields
            "xarray-zarr": { # An xarray-zarr source as defined above
                "url": url,
                "param": ["surface_geopotential", "lsm"],
            }
        }
    }
}

In [285]:
forcings = {
    "forcings": {
        "template": "${input.join.1.xarray-zarr}", # Path to another source in the recipe dictionary.
        "param": [
            "cos_latitude",
            "cos_longitude",
            "sin_latitude",
            "sin_longitude",
            "julian_day",
            "insolation",
        ]
    }
}

In [286]:
# We can combine all these datasets by joining them using the join key in a dictionary
recipe["input"] = {
    "join": [
        constant_variables,
        upper_air_variables,
        surface_variables,
        forcings
    ]
}

In [287]:
build = {
    "group_by": "daily", 
    "variable_naming": "param_levelist",
}

recipe["build"] = build

In [288]:
#I ran this in terminal, not here, but the command is below
print(
    yaml.dump(
        recipe,
        default_flow_style=False
    )
)

build:
  group_by: daily
  variable_naming: param_levelist
dates:
  end: '2024-01-31T21:00:00'
  frequency: 3h
  start: '2024-01-01T00:00:00'
description: Toy dataset for wind power prediction in BOZ
input:
  join:
  - repeated-dates:
      mode: constant
      source:
        xarray-zarr:
          param:
          - surface_geopotential
          - lsm
          url: cerra_jan2024_boz.zarr
  - xarray-zarr:
      param:
      - r
      - t
      - u
      - v
      - z
      url: cerra_jan2024_boz.zarr
  - xarray-zarr:
      param:
      - mcc
      - msl
      - synthetic_windpower
      - t2m
      - wdir10_cos
      - wdir10_sin
      - wdir_cos100
      - wdir_cos150
      - wdir_cos200
      - wdir_cos50
      - wdir_sin100
      - wdir_sin150
      - wdir_sin200
      - wdir_sin50
      - ws10
      - ws100
      - ws150
      - ws200
      - ws50
      url: cerra_jan2024_boz.zarr
  - forcings:
      param:
      - cos_latitude
      - cos_longitude
      - sin_latitude
      - 

In [289]:
name = recipe["name"]
recipe_path = f"{name}.yaml"
with open(recipe_path, 'w') as file:
    yaml.dump(recipe, file, default_flow_style=False)
dataset_path = "Cerra_jan24_Anemoi.zarr"

In [290]:
!anemoi-datasets create --overwrite {recipe_path} {dataset_path}

2025-10-15 14:52:29 INFO 🎬 Task init((),{}) starting
/mnt/data/weatherloss/WindPower/WindPower/lib/python3.12/site-packages/anemoi/utils/config.py:209: UserWarning: Modifying an instance of DotDict(). This class is intended to be immutable.
  warnings.warn("Modifying an instance of DotDict(). This class is intended to be immutable.")
2025-10-15 14:52:29 INFO Setting flatten_grid=True in config
2025-10-15 14:52:29 INFO Setting ensemble_dimension=2 in config
2025-10-15 14:52:29 INFO Setting flatten_grid=True in config
2025-10-15 14:52:29 INFO Setting ensemble_dimension=2 in config
2025-10-15 14:52:29 INFO {'end': '2024-01-31T21:00:00', 'frequency': '3h', 'start': '2024-01-01T00:00:00', 'group_by': 'daily'}
2025-10-15 14:52:29 INFO Groups(dates=31,StartEndDates(2024-01-01 00:00:00..2024-01-31 21:00:00 every 3:00:00))
2025-10-15 14:52:29 INFO Groups: Groups(dates=31,StartEndDates(2024-01-01 00:00:00..2024-01-31 21:00:00 every 3:00:00))
/mnt/data/weatherloss/WindPower/WindPower/lib/python3.

In [291]:
!anemoi-datasets inspect Cerra_jan24_Anemoi.zarr

📦 Path          : Cerra_jan24_Anemoi.zarr
🔢 Format version: 0.30.0

📅 Start      : 2024-01-01 00:00
📅 End        : 2024-01-31 21:00
⏰ Frequency  : 3h
🚫 Missing    : 0
🌎 Resolution : None
🌎 Field shape: [157, 211]

📐 Shape      : 248 × 72 × 1 × 33,127 (2.2 GiB)
💽 Size       : 1.2 GiB (1.2 GiB)
📁 Files      : 287

   Index │ Variable             │        Min │      Max │        Mean │      Stdev
   ──────┼──────────────────────┼────────────┼──────────┼─────────────┼───────────
       0 │ cos_latitude         │   0.548867 │ 0.666868 │    0.606946 │  0.0310695
       1 │ cos_longitude        │   0.983806 │        1 │    0.995883 │ 0.00412394
       2 │ insolation           │          0 │ 0.379927 │   0.0621947 │  0.0973029
       3 │ julian_day           │          0 │   24.625 │     12.3125 │    7.14462
       4 │ lsm                  │          0 │        1 │    0.526356 │   0.487836
       5 │ mcc                  │          0 │      100 │       34.21 │    39.3215
       6 │ msl        